# Run COMPASS GAM stages (R environment)

This notebook isolates all `mgcv`/R work from the Python run notebooks. Select an R conda kernel containing `mgcv` and `data.table`.

Required order:

1. Run the Python local notebook once to build prediction inputs.
2. Run **Stage A** here to create trajectory features.
3. Return to the Python notebook, set `REBUILD_PREDICTION_INPUTS = False`, and rerun its Stage 3 cell. This refits Python models with the GAM features without deleting them.
4. Return here and run **Stage B** for Cox nonlinearity tests.

Stage B rejects feature-selection files older than the trajectory features, guarding against accidentally testing the pre-GAM Python feature list.


In [ ]:
suppressPackageStartupMessages(library(data.table))

PROJECT_ROOT <- "/data/gusev/USERS/jpconnor/code/CAIA"
SURVIVAL_DIR <- file.path(PROJECT_ROOT, "COMPASS", "survival_analysis")

# Choose "profile_data" for the merged Parquets or "baseline" for ALL_2025_03.
DATA_VARIANT <- "profile_data"
ARM <- "adt"
LANDMARK_DAYS <- c(0L, 90L, 180L)
FORCE_RERUN <- TRUE

# --n-workers passed to both R scripts' mclapply dispatch. Do not default to
# parallel::detectCores() from R -- the cluster may allocate fewer cores than
# the node reports; pick this to match the actual job allocation.
N_WORKERS <- 8L

DATA_ROOT <- switch(
  DATA_VARIANT,
  profile_data = "/data/gusev/USERS/jpconnor/data/CAIA/COMPASS_PROFILE_DATA",
  baseline = "/data/gusev/USERS/jpconnor/data/CAIA/COMPASS",
  stop(sprintf("Unknown DATA_VARIANT: %s", DATA_VARIANT))
)
INPUTS_DIR <- file.path(DATA_ROOT, "survival_analysis", sprintf("prediction_inputs_%s", ARM))
MODEL_OUTPUT_DIR <- file.path(DATA_ROOT, "survival_analysis", sprintf("local_runs_%s", ARM))
NONLINEAR_OUTPUT_DIR <- file.path(MODEL_OUTPUT_DIR, "cox", "gam_nonlinearity")
RSCRIPT <- file.path(R.home("bin"), "Rscript")

# wait = TRUE (default): blocks and returns the exit status, as before.
# wait = FALSE: launches the subprocess in the background and returns its PID
# immediately, so the caller can spawn several before waiting on any of them
# (used by Stage B to run all landmarks concurrently instead of serially).
run_r_script <- function(script_name, args, wait = TRUE) {
  script_path <- file.path(SURVIVAL_DIR, script_name)
  if (!file.exists(script_path)) stop(sprintf("Missing R script: %s", script_path))
  command_args <- shQuote(c(script_path, args))
  cat(sprintf("[run ] %s %s\n", RSCRIPT, paste(command_args, collapse = " ")))
  if (wait) {
    status <- system2(RSCRIPT, args = command_args, stdout = "", stderr = "", wait = TRUE)
    if (!identical(status, 0L)) stop(sprintf("%s failed with status %s", script_name, status))
    return(invisible(status))
  }
  pid <- system2(RSCRIPT, args = command_args, stdout = "", stderr = "", wait = FALSE)
  invisible(pid)
}

# Blocks on a PID returned by run_r_script(..., wait = FALSE) via `wait(2)`
# (POSIX-only, matches this cluster). Returns the child's exit status.
wait_for_pid <- function(pid) {
  system2("bash", args = c("-c", shQuote(sprintf("while kill -0 %d 2>/dev/null; do sleep 1; done", pid))))
  # The exit status of the finished child itself isn't recoverable after the
  # fact this way; callers instead check for the expected output file, same
  # as the pre-existing FORCE_RERUN skip logic already does.
  invisible(NULL)
}

cat(sprintf("R:                 %s\n", R.version.string))
cat(sprintf("Rscript:           %s\n", RSCRIPT))
cat(sprintf("data variant:      %s\n", DATA_VARIANT))
cat(sprintf("prediction inputs: %s\n", INPUTS_DIR))
cat(sprintf("model outputs:     %s\n", MODEL_OUTPUT_DIR))
cat(sprintf("n workers:         %d\n", N_WORKERS))


## Stage A — hierarchical trajectory GAM features

Run this after `build_prediction_inputs` has created the pre-treatment long tables and canonical-lab file.


In [ ]:
required_inputs <- c(
  file.path(INPUTS_DIR, "canonical_labs_train_val.csv"),
  file.path(INPUTS_DIR, sprintf("pre_treatment_lab_long_landmark%d.csv", LANDMARK_DAYS))
)
missing_inputs <- required_inputs[!file.exists(required_inputs)]
if (length(missing_inputs)) {
  stop(sprintf("Missing Python-built input(s):\n%s", paste(missing_inputs, collapse = "\n")))
}

trajectory_outputs <- file.path(
  INPUTS_DIR, sprintf("gam_trajectory_features_landmark%d.csv", LANDMARK_DAYS)
)
diagnostic_outputs <- file.path(
  INPUTS_DIR, sprintf("gam_fit_diagnostics_landmark%d.csv", LANDMARK_DAYS)
)
curve_outputs <- file.path(
  INPUTS_DIR, sprintf("gam_trajectory_curves_landmark%d.csv", LANDMARK_DAYS)
)
stage_a_outputs <- c(trajectory_outputs, diagnostic_outputs, curve_outputs)
if (FORCE_RERUN || !all(file.exists(stage_a_outputs))) {
  stage_a_timing <- system.time(
    run_r_script(
      "gam_trajectory_features.R",
      c(
        "--inputs-dir", INPUTS_DIR,
        "--landmark-days", paste(LANDMARK_DAYS, collapse = ","),
        "--k-pop", "10",
        "--k-pat", "5",
        "--max-fs-patients", "0",
        "--patient-ridge-lambda", "5",
        "--trailing-window-days", "180",
        "--nthreads", "1",
        "--fit-split", "all",
        "--n-workers", as.character(N_WORKERS),
        "--curve-labs", "PSA,Testosterone",
        "--curve-grid-points", "9",
        "--auc-grid-points", "9"
      )
    )
  )
  cat(sprintf("[timing] Stage A wall time: %.1fs\n", stage_a_timing[["elapsed"]]))
} else {
  cat("[skip] all trajectory feature and diagnostic files already exist\n")
}
stopifnot(all(file.exists(stage_a_outputs)))
file.info(trajectory_outputs)[, c("size", "mtime"), drop = FALSE]

# Step 0 measurement: fit_seconds is measured tightly around each bam() call,
# excluding predict/CJ/merge/fwrite overhead, so stage_a_timing["elapsed"]
# minus this sum estimates that overhead directly.
diagnostics_all <- rbindlist(lapply(diagnostic_outputs, fread))
cat(sprintf(
  "[timing] sum(fit_seconds) across all labs/landmarks: %.1fs (%.0f%% of wall time)\n",
  sum(diagnostics_all$fit_seconds, na.rm = TRUE),
  100 * sum(diagnostics_all$fit_seconds, na.rm = TRUE) / stage_a_timing[["elapsed"]]
))
print(diagnostics_all[, .N, by = basis_used])
cat(sprintf("[timing] median edf_pop: %.2f\n", median(diagnostics_all$edf_pop, na.rm = TRUE)))


## Python handoff — stop here

Return to the matching Python run notebook. Set `REBUILD_PREDICTION_INPUTS = False`, keep `FORCE_RERUN = True`, and rerun Stage 3. The Python univariate and multivariable models will then merge the trajectory feature CSVs automatically. Do not rerun `build_prediction_inputs`, because its cleanup deliberately removes old GAM feature files.


## Stage B — nonlinear Cox GAM tests

Run only after completing the Python handoff above. Each landmark uses the exact feature-selection CSV from its ordinary (`both`) univariate run.


In [ ]:
dir.create(NONLINEAR_OUTPUT_DIR, recursive = TRUE, showWarnings = FALSE)
n_landmarks <- length(LANDMARK_DAYS)
nonlinear_outputs <- character(n_landmarks)
selection_paths <- character(n_landmarks)
needs_run <- logical(n_landmarks)

# Hoisted validation: every landmark's inputs and staleness guard are checked
# up front, before anything is spawned, so a stale landmark 180 selection file
# fails fast instead of after landmarks 0 and 90 have already burned wall time.
for (i in seq_len(n_landmarks)) {
  landmark <- LANDMARK_DAYS[[i]]
  trajectory_path <- file.path(INPUTS_DIR, sprintf("gam_trajectory_features_landmark%d.csv", landmark))
  selection_path <- file.path(
    MODEL_OUTPUT_DIR, "cox", sprintf("landmark_%d", landmark), "both",
    "cox_agg_feature_selection.csv"
  )
  if (!file.exists(trajectory_path)) stop(sprintf("Missing %s; run Stage A first.", trajectory_path))
  if (!file.exists(selection_path)) stop(sprintf("Missing %s; rerun Python Stage 3 first.", selection_path))

  trajectory_mtime <- file.info(trajectory_path)$mtime
  selection_mtime <- file.info(selection_path)$mtime
  if (selection_mtime < trajectory_mtime) {
    stop(sprintf(
      paste0(
        "Landmark %d feature selection predates its GAM trajectory features. ",
        "Set REBUILD_PREDICTION_INPUTS = False and FORCE_RERUN = True, then rerun Python Stage 3."
      ),
      landmark
    ))
  }

  selection_paths[[i]] <- selection_path
  output_path <- file.path(
    NONLINEAR_OUTPUT_DIR, sprintf("gam_cox_nonlinearity_landmark%d.csv", landmark)
  )
  nonlinear_outputs[[i]] <- output_path
  needs_run[[i]] <- FORCE_RERUN || !file.exists(output_path)
}

# Concurrent spawn: each landmark is an independent subprocess with no shared
# state (separate input rows, separate output file), so all validated
# landmarks launch at once with wait = FALSE instead of running serially.
stage_b_timing <- system.time({
  pids <- vector("list", n_landmarks)
  for (i in seq_len(n_landmarks)) {
    landmark <- LANDMARK_DAYS[[i]]
    if (!needs_run[[i]]) {
      cat(sprintf("[skip] landmark +%dd nonlinear GAM output exists\n", landmark))
      next
    }
    pids[[i]] <- run_r_script(
      "gam_cox_nonlinearity.R",
      c(
        "--inputs-dir", INPUTS_DIR,
        "--output-dir", NONLINEAR_OUTPUT_DIR,
        "--landmark-days", as.character(landmark),
        "--feature-selection-csv", selection_paths[[i]],
        "--n-workers", as.character(N_WORKERS)
      ),
      wait = FALSE
    )
  }
  for (i in seq_len(n_landmarks)) {
    if (!is.null(pids[[i]])) wait_for_pid(pids[[i]])
  }
})
cat(sprintf("[timing] Stage B wall time (all landmarks concurrent): %.1fs\n", stage_b_timing[["elapsed"]]))

stopifnot(all(file.exists(nonlinear_outputs)))
n_features_per_landmark <- vapply(nonlinear_outputs, function(p) nrow(fread(p)), integer(1))
cat(sprintf(
  "[timing] Stage B wall / n_features: %s\n",
  paste(sprintf("landmark %d: %d features, %.3fs/feature", LANDMARK_DAYS,
                n_features_per_landmark, stage_b_timing[["elapsed"]] / n_features_per_landmark),
        collapse = "; ")
))


## Nonlinearity summary


In [ ]:
nonlinearity <- do.call(rbind, lapply(nonlinear_outputs, read.csv, check.names = FALSE))
flagged <- nonlinearity[
  !is.na(nonlinearity$q_lrt) & nonlinearity$q_lrt < 0.05 & nonlinearity$edf > 1.5,
  c("landmark_days", "feature", "edf", "p_lrt", "q_lrt", "delta_aic")
]
flagged[order(flagged$q_lrt), ]
